## Implementing Object Detection Model from scratch
- Pytorch -cpu
- Car Dataset

In [ ]:
#verfiy you cuda version before installing pytorch cuda
!nvidia-smi

In [ ]:
#requirements for the project
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [ ]:
!pip install torch-summary torchmetrics -qU

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import kagglehub
import logging
import shutil
import datetime
from typing import Literal, Dict, TypedDict

import torch
from torch.functional import F
from torch import jit
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter


from torchvision.datasets import ImageFolder
from torchvision import models
from torchvision.transforms import transforms
from torchvision.utils import make_grid
from torchvision.utils import draw_bounding_boxes
from torchvision.ops import nms

from pathlib import Path
from PIL import Image
from zipfile import ZipFile
from google.colab import files

In [ ]:
import logging

#setup logging utility function
def get_logger():
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    # Prevent propagation to the root logger to avoid duplicate messages
    logger.propagate = False
    if not logger.handlers:  # Check if handlers are not already set
        handler = logging.StreamHandler()
        # Include %(name)s in the formatter for clearer output
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger

logger = get_logger()

In [ ]:
import kagglehub
import os
import logging
import shutil # Added for moving directories

#utility function to download datafrom kagglehub

def download_data(kaggle_path:str,destination_dir:str):
    """Function to download dataset from kagglehub
    Args: (kaggle_path):str=Path to the dataset on kaggle
    (destination_dir):str='Where we want to store the downloaded data"""
    logger = get_logger()
    try:
        logger.info(f"Downloading data from {kaggle_path}")
        # kagglehub.dataset_download downloads and extracts the dataset,
        # returning the path to the root directory of the extracted files.
        downloaded_path = kagglehub.dataset_download(kaggle_path)
        logger.info(f"Dataset downloaded and extracted to {downloaded_path}")

        # Now, move the extracted contents from downloaded_path to destination_dir
        os.makedirs(destination_dir, exist_ok=True)
        for item in os.listdir(downloaded_path):
            s = os.path.join(downloaded_path, item)
            d = os.path.join(destination_dir, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d) # copy2 preserves metadata
        logger.info(f"Contents moved/copied from {downloaded_path} to {destination_dir}")

    except Exception as e:
        logger.error(f"Error while downloading data {e}")

In [ ]:
#test the doenload function
download_data("kailaspsudheer/tiny-object-detection","./data_2")

In [ ]:
#utility function to load data from directory
def load_from_directory(directory:str, train_ratio: float = 0.8):
    """Function to load data from directory and perform train/test split.
    Args:
        directory (str): Path to the directory containing the data.
        train_ratio (float): Ratio of data to be used for training (e.g., 0.8 for 80% train, 20% test).
    Returns:
        tuple: A tuple containing (train_dataset, test_dataset)
    """
    logger = get_logger()
    try:
        logger.info(f"Loading data from {directory}")
        full_dataset = ImageFolder(directory)
        logger.info(f"Full dataset loaded from {directory} with {len(full_dataset)} images.")

        total_size = len(full_dataset)
        if total_size < 2:
            logger.warning(f"Dataset size ({total_size}) is too small for a train/test split. "
                           f"Returning full dataset as training (if not empty), test_dataset as None.")
            return (full_dataset, None) if total_size > 0 else (None, None)

        train_size = int(train_ratio * total_size)
        # Ensure both train and test sets have at least one sample
        if train_size == 0:
            train_size = 1
        test_size = total_size - train_size
        if test_size == 0:
            train_size = total_size - 1
            test_size = 1

        train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
        logger.info(f"Dataset split into training ({len(train_dataset)} images) "
                    f"and testing ({len(test_dataset)} images).")
        return train_dataset, test_dataset
    except Exception as e:
        logger.error(f"Error while loading data from directory: {e}")
        return None, None

In [ ]:
#testing the load_from_directory function
train_dataset, test_dataset = load_from_directory("./data_2")

if train_dataset:
    print(f"Training dataset size: {len(train_dataset)}")
    print(f"Training dataset classes: {train_dataset.dataset.classes}")
if test_dataset:
    print(f"Test dataset size: {len(test_dataset)}")
    print(f"Test dataset classes: {test_dataset.dataset.classes}")
else:
    print("No test dataset was created (possibly due to small total size).")

In [ ]:
import random
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torchvision import transforms
from PIL import Image

##utility function to display and createa grid of 6 images randomly from the train dataset

def display_and_create_grid(dataset, num_images=6, save_grid:bool=False, normalize:bool=False):
    logger = get_logger()
    if not dataset:
        logger.warning("Dataset is empty or None. Cannot display images.")
        return

    if len(dataset) < num_images:
        logger.warning(f"Dataset has only {len(dataset)} images, displaying all of them instead of {num_images}.")
        num_images = len(dataset)

    # Select random images
    random_indices = random.sample(range(len(dataset)), num_images)
    selected_images = []

    # Retrieve images and convert to tensor if necessary
    # ImageFolder dataset returns (image, label), so we take only the image
    for i in random_indices:
        image, _ = dataset[i]
        if isinstance(image, Image.Image):
            # Convert PIL Image to tensor for make_grid
            to_tensor = transforms.ToTensor()
            image = to_tensor(image)
        selected_images.append(image)

    if not selected_images:
        logger.warning("No images were selected for display.")
        return

    # Create a grid of images
    grid = make_grid(selected_images,
                     nrow=int(num_images**0.5),
                     padding=5,
                     normalize=normalize,
                     scale_each=True)

    # Convert the grid tensor to a PIL Image or numpy array for matplotlib
    # make_grid outputs a tensor of shape (C, H, W). For matplotlib, we need (H, W, C) for color images.
    ndarr = grid.mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).to('cpu', torch.uint8).numpy()

    # Display the grid
    plt.figure(figsize=(10, 10))
    plt.imshow(ndarr)
    plt.axis('off')
    plt.title(f'{num_images} Random Images from Dataset')

    if save_grid:
        try:
            os.makedirs('image_grids', exist_ok=True)
            filename = f'image_grids/random_grid_{num_images}.png'
            plt.savefig(filename)
            logger.info(f"Image grid saved to {filename}")
        except Exception as e:
            logger.error(f"Error saving image grid: {e}")

    plt.show()


In [ ]:
display_and_create_grid(train_dataset)

In [ ]:
display_and_create_grid(train_dataset,normalize=True,num_images=2)

### Non-Maximum Suppression (NMS) Function

The `non_max_suppression` function is a critical utility in object detection for refining the output of a model. Object detection models often predict multiple overlapping bounding boxes for the same object, each with an associated confidence score.

This function addresses this redundancy by:
1.  **Taking input:** A set of predicted bounding boxes (`boxes`), their corresponding confidence `scores`, and an `iou_threshold`.
2.  **Using `torchvision.ops.nms`:** It leverages the highly optimized `nms` function from `torchvision.ops`, which efficiently filters out redundant boxes.
3.  **Returning indices:** It returns the indices of the bounding boxes that should be kept, effectively eliminating highly overlapping boxes with lower confidence scores.

This ensures that for each detected object, only the most confident and representative bounding box is retained, leading to cleaner and more accurate detection results.

In [ ]:
#non max suppression utility function
def non_max_suppression(boxes, scores, iou_threshold):
    """
    Performs Non-Maximum Suppression (NMS) on a set of bounding boxes.

    Args:
        boxes (torch.Tensor): A tensor of bounding boxes of shape (N, 4),
                              where N is the number of boxes, and each box
                              is represented as [x1, y1, x2, y2].
        scores (torch.Tensor): A tensor of confidence scores for each bounding box (N,).
        iou_threshold (float): The Intersection Over Union (IOU) threshold for suppression.

    Returns:
        torch.Tensor: A tensor of indices of the boxes to keep after NMS.
    """
    logger = get_logger()
    if not isinstance(boxes, torch.Tensor):
        boxes = torch.tensor(boxes, dtype=torch.float32)
    if not isinstance(scores, torch.Tensor):
        scores = torch.tensor(scores, dtype=torch.float32)

    if boxes.numel() == 0 or scores.numel() == 0:
        logger.warning("No boxes or scores provided for NMS. Returning empty tensor.")
        return torch.tensor([], dtype=torch.long)

    # torchvision.ops.nms expects boxes in (x1, y1, x2, y2) format
    keep_indices = nms(boxes, scores, iou_threshold)
    logger.info(f"NMS completed. Kept {len(keep_indices)} out of {len(boxes)} boxes.")
    return keep_indices

## Helper Function: Load Images with Annotations

This section defines a utility function to load images and their associated bounding box annotations from a specified directory and a JSON annotation file. This is crucial for training object detection models, which require not just the images, but also the ground-truth locations and classes of objects within them.


In [ ]:
import json
from PIL import Image
from typing import List, Dict, Tuple

# Define a TypedDict for better structure and type hints for annotations
# this acts like a schema to support the load_image_annotations

class AnnotatedImage(TypedDict):
    image_path: str
    image: Image.Image
    boxes: List[List[float]]  # List of [x_min, y_min, x_max, y_max]/ that is a list of list of boxes
    labels: List[str]
    file_name: str

def load_image_annotations(image_folder_path: str,
                           annotations_file_path: str) -> List[AnnotatedImage]:
    """
    Loads images and their corresponding annotations (bounding boxes and labels).

    Args:
        image_folder_path (str): Path to the directory containing the images.
        annotations_file_path (str): Path to the JSON file containing annotations.

    Returns:
        List[AnnotatedImage]: A list of dictionaries, where each dictionary contains
                               image path, the PIL Image object, bounding boxes, and labels.
    """
    logger = get_logger()
    annotations_list: List[AnnotatedImage] = []

    if not os.path.exists(image_folder_path):
        logger.error(f"Image folder not found: {image_folder_path}")
        return []

    if not os.path.exists(annotations_file_path):
        logger.error(f"Annotations file not found: {annotations_file_path}")
        return []

    try:
        with open(annotations_file_path, 'r') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        logger.error(f"Error decoding JSON from {annotations_file_path}: {e}")
        return []
    except Exception as e:
        logger.error(f"Error reading annotations file {annotations_file_path}: {e}")
        return []

    # the JSON structure has an 'annotations' key and each annotation has 'image_id', 'bbox', 'category_id'
    # And an 'images' key with 'id', 'file_name'
    # And a 'categories' key with 'id', 'name'

    images_meta = {img['id']: img['file_name'] for img in data.get('images', [])}
    categories = {cat['id']: cat['name'] for cat in data.get('categories', [])}

    # Group annotations by image_id
    grouped_annotations: Dict[int, List[Dict]] = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in grouped_annotations:
            grouped_annotations[img_id] = []
        grouped_annotations[img_id].append(ann)

    for img_id, anns in grouped_annotations.items():
        file_name = images_meta.get(img_id)
        if not file_name:
            logger.warning(f"Image ID {img_id} has annotations but no corresponding file name in 'images' metadata. Skipping.")
            continue

        image_path = os.path.join(image_folder_path, file_name)
        if not os.path.exists(image_path):
            logger.warning(f"Image file not found: {image_path}. Skipping annotations for this image.")
            continue

        try:
            img = Image.open(image_path).convert("RGB")
        except Exception as e:
            logger.error(f"Error loading image {image_path}: {e}. Skipping.")
            continue

        boxes: List[List[float]] = []
        labels: List[str] = []

        for ann in anns:
            bbox = ann['bbox'] # [x, y, width, height] format
            category_id = ann['category_id']

            # Convert [x, y, width, height] to [x_min, y_min, x_max, y_max]
            x_min, y_min, width, height = bbox
            x_max = x_min + width
            y_max = y_min + height
            boxes.append([x_min, y_min, x_max, y_max])

            label_name = categories.get(category_id, 'unknown')
            labels.append(label_name)

        annotations_list.append({
            'image_path': image_path,
            'image': img,
            'boxes': boxes,
            'labels': labels,
            'file_name': file_name
        })
    logger.info(f"Successfully loaded {len(annotations_list)} images with annotations.")
    return annotations_list

In [ ]:
#test the annotations function
train_annotations = load_image_annotations(
    image_folder_path='./data_2/SkyFusion/train',
    annotations_file_path='./data_2/SkyFusion/train/_annotations.coco.json'
)

logger.info(f"Loaded {len(train_annotations)} annotated images for the training set.")

# Display some information from the first few annotated images
if train_annotations:
    logger.info("Sample of loaded training annotations:")
    for i, ann_data in enumerate(train_annotations[:3]): # Display details for first 3
        logger.info(f"  \nImage {i+1}: {ann_data['file_name']}, Boxes: {len(ann_data['boxes'])}, Labels: {ann_data['labels']}")

### Visualize Annotations

Now that we have loaded the image annotations, let's create a utility function to visualize these annotations on a given number of images from the dataset. This will help us verify that the bounding boxes and labels are correctly parsed and associated with their respective images.

In [ ]:
import random
from torchvision.utils import make_grid
from torchvision.transforms import ToTensor

def load_and_display_annotated_images(
    annotated_images: List[AnnotatedImage],
    num_images_to_display: int = 5,
    display: bool = True
) -> List[torch.Tensor]:
    """
    Loads a specified number of random images from a list of annotated images,
    draws bounding boxes on them, and optionally displays them.

    Args:
        annotated_images (List[AnnotatedImage]): A list of AnnotatedImage dictionaries.
        num_images_to_display (int): The number of images to randomly select and display.
        display (bool): If True, the images will be displayed using matplotlib.

    Returns:
        List[torch.Tensor]: A list of PyTorch Tensors of the images with drawn bounding boxes.
    """
    logger = get_logger()
    if not annotated_images:
        logger.warning("No annotated images provided.")
        return []

    if num_images_to_display > len(annotated_images):
        logger.warning(
            f"Requested {num_images_to_display} images, but only {len(annotated_images)} are available. "
            "Displaying all available images."
        )
        num_images_to_display = len(annotated_images)

    # Randomly select images to display
    random_indices = random.sample(range(len(annotated_images)), num_images_to_display)
    images_with_boxes = []
    to_tensor = ToTensor()

    for idx in random_indices:
        ann_data = annotated_images[idx]
        image_pil = ann_data['image']
        boxes = ann_data['boxes']
        labels = ann_data['labels']
        file_name = ann_data['file_name']

        # Convert PIL Image to a PyTorch Tensor
        image_tensor = to_tensor(image_pil)
        # Convert to uint8 for draw_bounding_boxes function
        image_tensor = (image_tensor * 255).to(torch.uint8)

        if boxes:
            # Convert boxes to tensor (draw_bounding_boxes expects tensor)
            # Ensure boxes are in (xmin, ymin, xmax, ymax) and are integers
            boxes_tensor = torch.tensor(boxes, dtype=torch.float)
            # Optionally, you can pass labels if you want them drawn on the image
            # For simplicity, we are just drawing boxes here
            annotated_image = draw_bounding_boxes(
                image_tensor,
                boxes_tensor,
                labels=labels,
                colors="red",
                width=2
            )
        else:
            annotated_image = image_tensor

        images_with_boxes.append(annotated_image)
        logger.info(f"Processed image: {file_name} with {len(boxes)} boxes.")

    if display and images_with_boxes:
        grid = make_grid(images_with_boxes, nrow=min(num_images_to_display, 4), padding=5)
        ndarr = grid.permute(1, 2, 0).cpu().numpy()

        plt.figure(figsize=(15, 15))
        plt.imshow(ndarr)
        plt.title(f"Random {num_images_to_display} Annotated Images")
        plt.axis('off')
        plt.show()

    return images_with_boxes

Now, let's test our `load_and_display_annotated_images` function to see the annotations on a few training images.

In [ ]:
# Display 3 random annotated images from the training set (ground truth)
_ = load_and_display_annotated_images(train_annotations, num_images_to_display=3)


### Determine Number of Classes

To properly initialize the object detection model, we need to know the total number of distinct object classes it needs to detect. We'll extract these from the `train_annotations` and add 1 for the background class (which is standard practice in object detection frameworks like PyTorch's Faster R-CNN).

In [ ]:
def get_num_classes(annotations: List[AnnotatedImage]) -> Tuple[int, List[str]]:
    """
    Calculates the number of unique classes from annotations.

    Args:
        annotations (List[AnnotatedImage]): A list of AnnotatedImage dictionaries.

    Returns:
        Tuple[int, List[str]]: A tuple containing the total number of classes
                               (including background) and a sorted list of class names.
    """
    all_labels = set() #create a set and ensure no repetitions
    for ann_data in annotations:
        for label in ann_data['labels']:
            all_labels.add(label)

    # Sort the labels for consistent mapping if needed later
    class_names = sorted(list(all_labels))
    # Add 1 for the background class, which is standard for most object detection models
    num_classes = len(class_names) + 1

    logger.info(f"Detected classes: {class_names}")
    logger.info(f"Total number of classes (including background): {num_classes}")
    return num_classes, class_names

In [ ]:
# Call the function to get the num_classes and class_names
num_classes, class_names = get_num_classes(train_annotations)

In [ ]:
import torch.nn as nn
from typing import Tuple, Dict, Any

def get_detection_loss_functions(
    cls_loss_name: str = 'CrossEntropyLoss',
    reg_loss_name: str = 'SmoothL1Loss',
    cls_loss_hparams: Dict[str, Any] = None,
    reg_loss_hparams: Dict[str, Any] = None,
) -> Tuple[nn.Module, nn.Module]:
    """
    Initializes and returns common loss functions for object detection.

    Args:
        cls_loss_name (str): Name of the classification loss function (e.g., 'CrossEntropyLoss', 'BCEWithLogitsLoss').
        reg_loss_name (str): Name of the regression loss function (e.g., 'SmoothL1Loss', 'MSELoss').
        cls_loss_hparams (Dict[str, Any]): Hyperparameters for the classification loss.
        reg_loss_hparams (Dict[str, Any]): Hyperparameters for the regression loss.

    Returns:
        Tuple[nn.Module, nn.Module]: A tuple containing the classification loss and the regression loss.
    """
    logger = get_logger()

    # Default hyperparameters
    if cls_loss_hparams is None:
        cls_loss_hparams = {}
    if reg_loss_hparams is None:
        reg_loss_hparams = {}

    # Initialize the classification loss
    if cls_loss_name.lower() in ['crossentropy', 'crossentropyoss', 'crossentropyloss']: #allow for some user typos
        cls_loss_fn = nn.CrossEntropyLoss(**cls_loss_hparams) #** to unzip the dict
    elif cls_loss_name.lower() == 'bcewithlogitsloss':
        cls_loss_fn = nn.BCEWithLogitsLoss(**cls_loss_hparams)
    else:
        raise ValueError(f"Unsupported classification loss: {cls_loss_name}")
    logger.info(f"Classification loss ({cls_loss_name}) initialized with parameters: {cls_loss_hparams}")

    # Initialize the regression loss
    if reg_loss_name.lower() == 'smoothl1loss':
        reg_loss_fn = nn.SmoothL1Loss(**reg_loss_hparams)
    elif reg_loss_name.lower() == 'mseloss':
        reg_loss_fn = nn.MSELoss(**reg_loss_hparams)
    else:
        raise ValueError(f"Unsupported regression loss: {reg_loss_name}")
    logger.info(f"Regression loss ({reg_loss_name}) initialized with parameters: {reg_loss_hparams}")

    return cls_loss_fn, reg_loss_fn

In [ ]:
# Instantiate the classification and regression loss functions
cls_loss_fn, reg_loss_fn = get_detection_loss_functions(
    cls_loss_name='CrossEntropyLoss',
    reg_loss_name='SmoothL1Loss',
    cls_loss_hparams={'reduction': 'none'},
    #reg_loss_hparams={'reduction': 'mean'}
)

logger.info("Detection loss functions setup completed.")

## Model Definition Step
- include basic block for the residual conections
- the ssd head for the model predictions
- the final model that encapdulates all the above

In [ ]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    """A standard residual block with two 3x3 convolutions."""
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [ ]:
import torch
import torch.nn as nn

class CustomBackbone(nn.Module):
    """
    This is a  tiny ResNet-like backbone that outputs feature maps at three scales:
      - scale_8: 1/8 of input spatial size
      - scale_16: 1/16 of input spatial size
      - scale_32: 1/32 of input spatial size
    Number of output channels: [64, 128, 256] (can be changed easily).
    """
    def __init__(self):
        super().__init__()
        self.in_planes = 32

        # Initial stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        # Instantiate MaxPool2d in __init__ to enable JIT scripting
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        # Stage 1: 1/2 -> 1/4
        self.layer1 = self._make_layer(32, 2, stride=1)   # output size still 1/4
        # Stage 2: 1/4 -> 1/8
        self.layer2 = self._make_layer(64, 2, stride=2)   # now 1/8
        # Stage 3: 1/8 -> 1/16
        self.layer3 = self._make_layer(128, 2, stride=2)  # 1/16
        # Stage 4: 1/16 -> 1/32
        self.layer4 = self._make_layer(256, 2, stride=2)  # 1/32

        # We'll extract the outputs after layer2, layer3, layer4
        self.out_channels = [64, 128, 256]

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        """
        Returns a list of feature maps:
          [scale_8 (1/8), scale_16 (1/16), scale_32 (1/32)]
        """
        x = self.stem(x)          # 1/2
        x = self.maxpool(x)       # 1/4
        scale_4 = self.layer1(x)  # still 1/4
        scale_8 = self.layer2(scale_4)   # 1/8
        scale_16 = self.layer3(scale_8)  # 1/16
        scale_32 = self.layer4(scale_16) # 1/32
        return (scale_8, scale_16, scale_32)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import product

class SSDHead(nn.Module):
    """
    One detection head applied to a feature map.
    It consists of two parallel 3x3 convolutions:
      - loc_head: predicts 4 deltas per anchor box
      - cls_head: predicts class scores per anchor box (including background)
    """
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.num_anchors = num_anchors
        self.num_classes = num_classes # Fix: Store num_classes as an instance attribute
        self.loc_head = nn.Conv2d(in_channels, num_anchors * 4, kernel_size=3, padding=1)
        self.cls_head = nn.Conv2d(in_channels, num_anchors * (num_classes + 1), kernel_size=3, padding=1)

    def forward(self, x):
        batch_size, _, h, w = x.size()
        # locations: (B, A*4, H, W) -> (B, H*W*A, 4)
        loc = self.loc_head(x).permute(0, 2, 3, 1).contiguous().view(batch_size, -1, 4)
        # class logits: (B, A*(C+1), H, W) -> (B, H*W*A, C+1)
        cls = self.cls_head(x).permute(0, 2, 3, 1).contiguous().view(batch_size, -1, self.num_anchors, self.num_classes+1)
        cls = cls.view(batch_size, -1, self.num_classes + 1)  # (B, total_anchors, C+1)
        return loc, cls

In [ ]:
#final model
import torch.nn as nn

class ObjDet_V1(nn.Module):
    """
    Custom object detector from scratch.
    Backbone: our tiny ResNet variant (multi‑scale outputs).
    Heads: SSD‑style box and class predictors.
    The forward pass returns decoded boxes, class labels, and confidence scores.
    """
    def __init__(self, num_classes, input_size=512):
        """
        Args:
            num_classes: number of object classes (does not include the background).
            input_size: assumed square image size (used only for anchor generation).
        """
        super().__init__()

        self.num_classes = num_classes
        self.backbone = CustomBackbone()
        out_channels = self.backbone.out_channels  # [64, 128, 256]

        # Anchor configuration per scale:
        # Each dict defines: feature map stride, sizes (in pixels) and aspect ratios.
        self.feature_maps = [
            {"stride": 8,  "sizes": [30, 60],   "ratios": [1, 2, 0.5]},                     # scale_8
            {"stride": 16, "sizes": [90, 120],  "ratios": [1, 2, 0.5, 3, 0.33]},            # scale_16
            {"stride": 32, "sizes": [150, 200], "ratios": [1, 2, 0.5]}                      # scale_32
        ]

        # Build heads for each scale
        self.heads = nn.ModuleList()
        self.num_anchors_per_loc = []
        for fm, in_ch in zip(self.feature_maps, out_channels):
            num_anchors = len(fm["sizes"]) * len(fm["ratios"])
            self.num_anchors_per_loc.append(num_anchors)
            self.heads.append(SSDHead(in_ch, num_anchors, num_classes))

        # Pre‑compute all anchor boxes (image coordinates) and store as buffer
        self.input_size = input_size
        anchors = self._generate_anchors()
        self.register_buffer("anchors", anchors)  # shape (total_anchors, 4)

    def _generate_anchors(self):
        """
        Build anchor boxes for every feature map location according to the
        defined strides, sizes and ratios. Coordinates are in (x1, y1, x2, y2)
        relative to the full input image (0..input_size).
        """
        all_anchors = []
        for fm in self.feature_maps:
            stride = fm["stride"]
            sizes = fm["sizes"]
            ratios = fm["ratios"]
            fm_h = fm_w = self.input_size // stride
            # Pre‑compute box templates (half width & height) for [x1,y1,x2,y2]
            boxes = []
            for s in sizes:
                for r in ratios:
                    w = s * math.sqrt(r)
                    h = s / math.sqrt(r)
                    # box in (cx, cy, w, h) -> (x1,y1,x2,y2) with (0,0) center
                    boxes.append([-w/2, -h/2, w/2, h/2])
            boxes = torch.tensor(boxes, dtype=torch.float32)  # (A, 4) in cx-centered half-sizes

            # Create grid of centres
            grid_y, grid_x = torch.meshgrid(
                torch.arange(fm_h, dtype=torch.float32),
                torch.arange(fm_w, dtype=torch.float32),
                indexing='ij'
            )
            # centre coordinates in image pixels (0.5 offset to pixel centre)
            cy = (grid_y + 0.5) * stride
            cx = (grid_x + 0.5) * stride
            cy = cy.reshape(-1, 1)  # (H*W, 1)
            cx = cx.reshape(-1, 1)
            centres = torch.cat([cx, cy, cx, cy], dim=1)  # (H*W, 4) to add to boxes

            # Broadcast: (H*W, A, 4)
            anchors_fm = centres.unsqueeze(1) + boxes.unsqueeze(0)  # (H*W, A, 4)
            anchors_fm = anchors_fm.reshape(-1, 4)  # (H*W*A, 4)
            all_anchors.append(anchors_fm)

        return torch.cat(all_anchors, dim=0)  # (total_anchors, 4)

    def _decode_boxes(self, loc_preds):
        """
        Decode raw bounding box predictions into (x1,y1,x2,y2) image coordinates.
        loc_preds: (B, total_anchors, 4) – predicted offsets in SSD style
                    (delta_cx, delta_cy, delta_w, delta_h) w.r.t. anchor centre and size.
        """
        anchors = self.anchors  # (N, 4)
        # Convert anchor from (x1,y1,x2,y2) to (cx, cy, w, h)
        anchor_w = anchors[:, 2] - anchors[:, 0]
        anchor_h = anchors[:, 3] - anchors[:, 1]
        anchor_cx = (anchors[:, 0] + anchors[:, 2]) / 2.0
        anchor_cy = (anchors[:, 1] + anchors[:, 3]) / 2.0

        # Predicted deltas
        dx = loc_preds[:, :, 0]
        dy = loc_preds[:, :, 1]
        dw = loc_preds[:, :, 2]
        dh = loc_preds[:, :, 3]

        # Decode
        pred_cx = dx * anchor_w + anchor_cx
        pred_cy = dy * anchor_h + anchor_cy
        pred_w = torch.exp(dw) * anchor_w
        pred_h = torch.exp(dh) * anchor_h

        # Back to (x1,y1,x2,y2)
        x1 = pred_cx - pred_w / 2.0
        y1 = pred_cy - pred_h / 2.0
        x2 = pred_cx + pred_w / 2.0
        y2 = pred_cy + pred_h / 2.0

        return torch.stack([x1, y1, x2, y2], dim=2)  # (B, N, 4)

    def forward(self, images):
        """
        images: (B, 3, H, W) with H=W=self.input_size ideally.

        Returns dict:
            "boxes":  decoded bounding boxes in (x1,y1,x2,y2) format (pixel coords).
            "labels": predicted class index (0 = background, 1..C = object classes).
            "scores": confidence score for the predicted class (max probability after softmax).
            If in training mode, also returns raw 'loc_preds' and 'cls_logits' for loss calculation.
        """
        features = self.backbone(images)  # list of [feat_8, feat_16, feat_32]

        locs, clss = [], []
        for feat, head in zip(features, self.heads):
            l, c = head(feat)  # (B, H*W*A, 4), (B, H*W*A, C+1)
            locs.append(l)
            clss.append(c)

        # Concatenate all anchors from all scales
        loc_preds = torch.cat(locs, dim=1)   # (B, total_anchors, 4)
        cls_logits = torch.cat(clss, dim=1)  # (B, total_anchors, C+1)

        # Decode boxes
        decoded_boxes = self._decode_boxes(loc_preds)

        # Softmax to get probabilities, then pick top class and score
        probs = F.softmax(cls_logits, dim=-1)  # (B, N, C+1)
        scores, labels = torch.max(probs, dim=-1)  # (B, N)

        output = {
            "boxes": decoded_boxes,
            "labels": labels,
            "scores": scores,
            "loc_preds": loc_preds,  # Always return raw predictions
            "cls_logits": cls_logits # Always return raw predictions
        }

        return output

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger = get_logger()
logger.info(f'On device: {device}')
# Initialize ObjDet_V1 with the number of *object* classes, not including background.
# The get_num_classes function returns 'num_classes' as total (including background)
# and 'class_names' as the list of object classes.
model = ObjDet_V1(num_classes=len(class_names)).to(device)

#setup dummy images
dummy_img = torch.randn(2, 3, 512, 512).to(device) #(batch_size,channels,height,width)

#perform inference on the untrained model with 2 dummy images
out = model(dummy_img)
print(out['boxes'].shape)   # torch.Size([2, total_anchors, 4])
print(out['labels'].shape)  # torch.Size([2, total_anchors])
print(out['scores'].shape)  # torch.Size([2, total_anchors])

In [ ]:
import random
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor
from torchvision.utils import draw_bounding_boxes, make_grid

# Define the transformation for input images to the model
# Assuming the model expects normalized tensors
transform = transforms.Compose([
    transforms.Resize((model.input_size, model.input_size)), # Resize to model's expected input size
    transforms.ToTensor(),
    # ypu can add normalization if the model was trained with it; add it like this below:
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#function to display model predictions
def display_model_predictions(
    model,
    annotated_images: List[AnnotatedImage],
    class_names: List[str],
    num_images_to_display: int = 3,
    confidence_threshold: float = 0.5,
    show_ground_truth: bool = False,
    apply_nms: bool = False, # parameter to apply NMS(Non MAx Suppression)
    nms_iou_threshold: float = 0.5 # parameter for NMS IOU threshold
):
    logger = get_logger()
    if not annotated_images:
        logger.warning("No annotated images provided.")
        return

    if num_images_to_display > len(annotated_images):
        num_images_to_display = len(annotated_images)

    random_indices = random.sample(range(len(annotated_images)), num_images_to_display)
    predicted_images = []

    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # this disables gradient calculations. same as torch.inference_mode()
        for idx in random_indices:
            ann_data = annotated_images[idx]
            image_pil = ann_data['image']
            file_name = ann_data['file_name']

            #1. Preprocess the image for the model
            image_tensor = transform(image_pil).to(device)
            # Add batch dimension
            image_batch = image_tensor.unsqueeze(0)

            #2. Get model predictions
            predictions = model(image_batch)

            #3. Extract and filter raw predictions based on confidence
            pred_boxes = predictions['boxes'][0] # Take first item from batch
            pred_labels = predictions['labels'][0]
            pred_scores = predictions['scores'][0]

            # Initial filtering by confidence threshold
            keep_preds_conf = pred_scores > confidence_threshold
            filtered_pred_boxes = pred_boxes[keep_preds_conf]
            filtered_pred_labels_raw = pred_labels[keep_preds_conf]
            filtered_pred_scores = pred_scores[keep_preds_conf]

            # If no boxes passed the confidence threshold AND threshold is 0.0 (for debugging untrained models),
            # show a few top-scoring predictions anyway to ensure some visualization.
            if filtered_pred_boxes.numel() == 0 and confidence_threshold == 0.0:
                logger.info(f"No predictions passed the 0.0 confidence threshold for {file_name}. Displaying top 100 highest scoring predictions for debugging visualization.")
                top_k_to_show = min(100, len(pred_scores)) # Limit to 100 boxes, or fewer if not many predictions
                if top_k_to_show > 0:
                    # Get top N scores, even if very low, to ensure *some* boxes are drawn.
                    top_scores, top_indices = torch.topk(pred_scores, top_k_to_show, largest=True)
                    filtered_pred_boxes = pred_boxes[top_indices]
                    filtered_pred_labels_raw = pred_labels[top_indices]
                    filtered_pred_scores = pred_scores[top_indices]
                else:
                    logger.warning(f"No predictions generated by the model for {file_name}.")

            # Apply NMS if needed, set to true
            if apply_nms and filtered_pred_boxes.numel() > 0:
                # NMS expects boxes in (x1, y1, x2, y2) and scores
                nms_keep_indices = non_max_suppression(
                    boxes=filtered_pred_boxes,
                    scores=filtered_pred_scores,
                    iou_threshold=nms_iou_threshold
                )
                #keep only indices where the nms is applied and chosen
                filtered_pred_boxes = filtered_pred_boxes[nms_keep_indices]
                filtered_pred_labels_raw = filtered_pred_labels_raw[nms_keep_indices]
                filtered_pred_scores = filtered_pred_scores[nms_keep_indices]
                logger.info(f"NMS applied to predictions for {file_name}. Kept {len(filtered_pred_boxes)} boxes.")

            # Prepare original image for drawing (uint8, C, H, W)
            # The image_tensor here is ALREADY resized to model.input_size
            image_to_draw = (image_tensor * 255).to(torch.uint8)

            all_boxes_to_draw = []
            all_labels_to_display = []
            all_colors = []

            # Add Ground Truth to show how well model performed in drawing boxes around the object compared to the actual
            if show_ground_truth:
                gt_boxes_raw = ann_data['boxes']
                gt_labels_raw = ann_data['labels']
                if gt_boxes_raw:
                    # Get original image dimensions from PIL image before transformation
                    original_width, original_height = image_pil.size

                    # Calculate scaling factors
                    scale_x = model.input_size / original_width
                    scale_y = model.input_size / original_height

                    # Scale ground truth boxes
                    scaled_gt_boxes = []
                    for bbox in gt_boxes_raw:
                        # Correct unpacking of bbox (it's already xmin, ymin, xmax, ymax)
                        x_min, y_min, x_max, y_max = bbox
                        scaled_gt_boxes.append([
                            x_min * scale_x,
                            y_min * scale_y,
                            x_max * scale_x,
                            y_max * scale_y
                        ])

                    # Move ground truth boxes to the same device as model predictions
                    gt_boxes_tensor = torch.tensor(scaled_gt_boxes, dtype=torch.float32).to(device)
                    all_boxes_to_draw.append(gt_boxes_tensor)
                    all_labels_to_display.extend([f"GT: {lbl}" for lbl in gt_labels_raw])
                    all_colors.extend(["red"] * len(gt_boxes_raw))
                logger.info(f"Processed ground truth for {file_name}. Detected {len(gt_boxes_raw)} objects.")


            # ===Add Model Predictions ===
            if filtered_pred_boxes.numel() > 0:
                pred_labels_text = []
                pred_boxes_for_display = []

                for label, score, box in zip(filtered_pred_labels_raw, filtered_pred_scores, filtered_pred_boxes):
                    # Filter out background predictions if confidence_threshold is not 0.0
                    # Background class has label 0.
                    if label.item() == 0 and confidence_threshold != 0.0:
                        continue

                    pred_boxes_for_display.append(box)
                    if label.item() == 0:
                        pred_labels_text.append(f"Pred: Background: {score.item():.2f}")
                    else:
                        # Ensure class_names index is valid (label.item() is 1-indexed for actual classes)
                        pred_labels_text.append(f"Pred: {class_names[label.item()-1]}: {score.item():.2f}")

                if pred_boxes_for_display:
                    all_boxes_to_draw.append(torch.stack(pred_boxes_for_display))
                    all_labels_to_display.extend(pred_labels_text)
                    all_colors.extend(["green"] * len(pred_boxes_for_display))
                    logger.info(f"Processed predictions for {file_name}. Displaying {len(pred_boxes_for_display)} objects (excluding background if threshold > 0). Total detected by model: {len(filtered_pred_boxes)}.")
                else:
                    logger.info(f"No relevant (non-background or above threshold) predictions to display for {file_name}.")


            final_drawn_image = image_to_draw
            if all_boxes_to_draw:
                combined_boxes_tensor = torch.cat(all_boxes_to_draw, dim=0)
                final_drawn_image = draw_bounding_boxes(
                    image_to_draw,
                    combined_boxes_tensor,
                    labels=all_labels_to_display,
                    colors=all_colors,
                    width=2
                )
            predicted_images.append(final_drawn_image)

    if predicted_images:
        grid = make_grid(predicted_images, nrow=num_images_to_display, padding=5)
        ndarr = grid.permute(1, 2, 0).cpu().numpy()

        plt.figure(figsize=(15, 15))
        plt.imshow(ndarr)
        title_text = ""
        if show_ground_truth:
            title_text += "Ground Truth (Red) & "
        title_text += f"Model Predictions (Green, Confidence > {confidence_threshold})"
        if apply_nms:
            title_text += f" with NMS (IoU > {nms_iou_threshold})"
        plt.title(title_text)
        plt.axis('off')
        plt.show()

In [ ]:
# Display 3 random images with ground truth, model predictions, and NMS applied
display_model_predictions(
    model=model,
    annotated_images=train_annotations,
    class_names=class_names,
    num_images_to_display=1,
    confidence_threshold=0.5, # A low threshold to get more predictions
    show_ground_truth=True,
    apply_nms=True,            # Apply NMS
    nms_iou_threshold=0.3      # NMS IoU threshold
)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor


class ObjectDetectionDataset(Dataset):
    """
    A custom PyTorch Dataset for object detection, handling image loading,
    transformations, bounding box scaling, and label encoding.
    """
    def __init__(self, annotated_images: List[AnnotatedImage], class_names: List[str], transform=None):
        self.annotated_images = annotated_images
        # Create a 1-indexed map for class names, reserving 0 for background
        self.class_names_map = {name: i + 1 for i, name in enumerate(class_names)}
        self.transform = transform

    def __len__(self):
        return len(self.annotated_images)

    def __getitem__(self, idx):
        ann_data = self.annotated_images[idx]
        image_pil = ann_data['image']
        gt_boxes = ann_data['boxes']  # [x_min, y_min, x_max, y_max]
        gt_labels_str = ann_data['labels']

        original_width, original_height = image_pil.size

        # Apply transformation (e.g., resize and ToTensor)
        if self.transform:
            image_transformed = self.transform(image_pil)
            # The transform should ideally include Resize, so we get target dimensions
            # If transform only does ToTensor, target_width/height will be original_width/height
            # Assuming CxHxW format for image_transformed
            target_height, target_width = image_transformed.shape[-2], image_transformed.shape[-1]

            scale_x = target_width / original_width
            scale_y = target_height / original_height

            scaled_gt_boxes = []
            for bbox in gt_boxes:
                x_min, y_min, x_max, y_max = bbox
                scaled_gt_boxes.append([
                    x_min * scale_x,
                    y_min * scale_y,
                    x_max * scale_x,
                    y_max * scale_y
                ])
            gt_boxes_tensor = torch.tensor(scaled_gt_boxes, dtype=torch.float32)
        else:
            # Default to ToTensor if no transform is provided
            image_transformed = ToTensor()(image_pil)
            gt_boxes_tensor = torch.tensor(gt_boxes, dtype=torch.float32)

        # Convert string labels to numerical IDs
        gt_labels_numeric = torch.tensor([self.class_names_map[label] for label in gt_labels_str], dtype=torch.long)

        return image_transformed, gt_boxes_tensor, gt_labels_numeric

logger.info("ObjectDetectionDataset class defined.")


In [ ]:
#collate function to handle variable numbers of objects per image
#helper function to
def collate_fn(batch):
    """
    Custom collate function for DataLoader to handle variable numbers of objects per image.
    Combines images into a batch tensor and keeps boxes/labels as lists of tensors.
    """
    images = [item[0] for item in batch]
    boxes = [item[1] for item in batch]
    labels = [item[2] for item in batch]
    return torch.stack(images, 0), boxes, labels
logger.info('collate_fn function defined')

In [ ]:
#creating the dataloader with the custom obj detection dataset and collate function
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
from torchvision.transforms import transforms

def get_object_detection_dataloader(
    annotated_images: List[AnnotatedImage],
    class_names: List[str],
    transform,
    batch_size: int = 4,
    shuffle: bool = True,
    num_workers: int = 2
) -> DataLoader:
    """
    Creates a DataLoader for object detection data.

    Args:
        annotated_images (List[AnnotatedImage]): List of annotated image dictionaries.
        class_names (List[str]): List of object class names.
        transform: Transformations to apply to the images (e.g., Resize, ToTensor).
        batch_size (int): Number of images per batch.
        shuffle (bool): Whether to shuffle the data at the beginning of each epoch.
        num_workers (int): How many subprocesses to use for data loading.

    Returns:
        DataLoader: A PyTorch DataLoader configured for object detection training.
    """
    dataset = ObjectDetectionDataset(annotated_images, class_names, transform)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_fn  # using the custom collate function to alloww the dataloader deal with vairiable number of objs in the image
    )
    logger.info(f"Created DataLoader with {len(dataset)} samples and batch size {batch_size}.")
    return dataloader

logger.info("get_object_detection_dataloader function defined")

In [ ]:
# Define the transformation for input images to the model
# This transform will be applied to both training and validation datasets
transforms_dataloader = transforms.Compose([
    transforms.Resize((model.input_size, model.input_size)), # Resize to model's expected input size
    transforms.ToTensor(),
])

#=== TESTING THE DATALOADER ===
# 1. Initialize Training DataLoader
logger.info("Initializing training DataLoader...")
train_dataloader = get_object_detection_dataloader(
    annotated_images=train_annotations,
    class_names=class_names,
    transform=transforms_dataloader,
    batch_size=16,
    shuffle=True,
    num_workers=2
)
logger.info("Training DataLoader initialized.")

# 2. Initialize Validation DataLoader (if data exists)
val_dataloader = None
try:
    logger.info("Attempting to load validation annotations...")
    #apply the load_image_annotation from the directory
    val_annotations = load_image_annotations(
        image_folder_path='./data_2/SkyFusion/valid/',
        annotations_file_path='./data_2/SkyFusion/valid/_annotations.coco.json'
    )
    if val_annotations:
        logger.info("Validation annotations loaded. Initializing validation DataLoader...")
        val_dataloader = get_object_detection_dataloader(
            annotated_images=val_annotations,
            class_names=class_names,
            transform=transforms_dataloader,
            batch_size=16,
            shuffle=False,
            num_workers=2
        )
        logger.info("Validation DataLoader initialized.")
    else:
        logger.warning("No validation annotations found or loaded. Validation DataLoader will not be created.")
except Exception as e:
    logger.error(f"Error loading validation annotations: {e}. Validation DataLoader will not be created.")

In [ ]:
# 3. Test the DataLoaders
logger.info("Testing training DataLoader...")
for images, boxes, labels in train_dataloader:
    logger.info(f"Train Batch: Images shape: {images.shape}, Number of boxes lists: {len(boxes)}, Number of labels lists: {len(labels)}\n")
    logger.info(f"Example Train Box list length: {len(boxes[0])}, Example Train Label list length: {len(labels[0])}")
    break # set break to take one batch

if val_dataloader:
    logger.info("Testing validation DataLoader...")
    for images, boxes, labels in val_dataloader:
        logger.info(f"Validation Batch: Images shape: {images.shape}, Number of boxes lists: {len(boxes)}, Number of labels lists: {len(labels)}\n")
        logger.info(f"Example Validation Box list length: {len(boxes[0])}, Example Validation Label list length: {len(labels[0])}")
        break # set break to take one batch
else:
    logger.warning("Validation DataLoader not available for testing.")

logger.info("Dataloader setup and test complete.")

In [ ]:
#more helper functions

import torch

def iou(box_a, box_b):
    """
    Calculate the Intersection Over Union (IoU) between two sets of bounding boxes.
    Args:
        box_a: (tensor) A `Tensor` of shape (A, 4) representing `A` bounding boxes.
        box_b: (tensor) A `Tensor` of shape (B, 4) representing `B` bounding boxes.
    Returns:
        (tensor) The IoU values for each pair of boxes, shape (A, B).
    """
    A = box_a.size(0)
    B = box_b.size(0)

    # Calculate intersection coordinates
    # For each box_a, broadcast with box_b to find max x1/y1 and min x2/y2
    max_xy = torch.min(box_a[:, 2:].unsqueeze(1).expand(A, B, 2), # x2, y2 of box_a and box_b
                       box_b[:, 2:].unsqueeze(0).expand(A, B, 2))
    min_xy = torch.max(box_a[:, :2].unsqueeze(1).expand(A, B, 2), # x1, y1 of box_a and box_b
                       box_b[:, :2].unsqueeze(0).expand(A, B, 2))

    inter = torch.clamp((max_xy - min_xy), min=0) # (A, B, 2) width and height of intersection
    inter_area = inter[:, :, 0] * inter[:, :, 1] # (A, B)

    # Calculate area of box_a and box_b
    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1])).unsqueeze(1).expand_as(inter_area) # (A, B)
    area_b = ((box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1])).unsqueeze(0).expand_as(inter_area) # (A, B)

    union_area = area_a + area_b - inter_area
    # Handle cases where union_area might be zero (e.g., no overlap at all)
    iou_val = inter_area / (union_area + 1e-8) # Add a small epsilon to prevent division by zero
    return iou_val


class BoxEncoder:
    """
    Encodes the variance of the bounding box offsets for localization loss.
    Adapted from SSD implementation
    This class will be responsible for encoding ground truth bounding boxes into the delta format (offsets, scale factors) relative to the anchor boxes.
    This is the format our ObjDet_V1 model's regression head (loc_head) predicts..
    """
    def __init__(self, variance=(0.1, 0.2)):
        self.variance = variance

    def encode(self, gt_boxes, anchors):
        """
        Encode ground truth boxes (gt_boxes) into deltas suitable for SSD regression loss,
        relative to given anchors.
        Args:
            gt_boxes: (tensor) Ground truth bounding boxes, shape (num_gt, 4) in (x1,y1,x2,y2).
            anchors: (tensor) Anchor boxes, shape (num_anchors, 4) in (x1,y1,x2,y2).
        Returns:
            (tensor) Encoded bounding box targets, shape (num_gt, 4).
        """
        # Convert to center-size coordinates for encoding
        # Anchors
        anchor_cx = (anchors[:, 0] + anchors[:, 2]) / 2
        anchor_cy = (anchors[:, 1] + anchors[:, 3]) / 2
        anchor_w = anchors[:, 2] - anchors[:, 0]
        anchor_h = anchors[:, 3] - anchors[:, 1]

        # Ground truths
        gt_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
        gt_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
        gt_w = gt_boxes[:, 2] - gt_boxes[:, 0]
        gt_h = gt_boxes[:, 3] - gt_boxes[:, 1]

        # Encode offsets
        # deltas (cx, cy, w, h)
        cx_target = (gt_cx - anchor_cx) / anchor_w / self.variance[0]
        cy_target = (gt_cy - anchor_cy) / anchor_h / self.variance[0]
        w_target = torch.log(gt_w / anchor_w) / self.variance[1]
        h_target = torch.log(gt_h / anchor_h) / self.variance[1]

        return torch.stack([cx_target, cy_target, w_target, h_target], dim=1)

    def decode(self, loc_preds, anchors):
        """
        Decode predicted deltas back into (x1,y1,x2,y2) bounding box coordinates.
        Args:
            loc_preds: (tensor) Predicted deltas, shape (num_predictions, 4).
            anchors: (tensor) Anchor boxes, shape (num_anchors, 4).
        Returns:
            (tensor) Decoded bounding boxes, shape (num_predictions, 4).
        """
        # Convert anchor to center-size
        anchor_cx = (anchors[:, 0] + anchors[:, 2]) / 2
        anchor_cy = (anchors[:, 1] + anchors[:, 3]) / 2
        anchor_w = anchors[:, 2] - anchors[:, 0]
        anchor_h = anchors[:, 3] - anchors[:, 1]

        # Predicted deltas
        dx = loc_preds[:, 0] * self.variance[0]
        dy = loc_preds[:, 1] * self.variance[0]
        dw = loc_preds[:, 2] * self.variance[1]
        dh = loc_preds[:, 3] * self.variance[1]

        # Decode center and size
        pred_cx = dx * anchor_w + anchor_cx
        pred_cy = dy * anchor_h + anchor_cy
        pred_w = torch.exp(dw) * anchor_w
        pred_h = torch.exp(dh) * anchor_h

        # Convert back to (x1,y1,x2,y2)
        x1 = pred_cx - pred_w / 2
        y1 = pred_cy - pred_h / 2
        x2 = pred_cx + pred_w / 2
        y2 = pred_cy + pred_h / 2

        return torch.stack([x1, y1, x2, y2], dim=1)


class MatchPrior:
    """
    Matches ground truth boxes to anchor boxes based on IoU.
    Assigns a ground truth class label and encoded box offset to each anchor.
    This is used to generate targets for training.
    This class will handle the crucial task of matching ground truth bounding boxes to our predefined anchor boxes.
    It determines which anchors are considered 'positive' (responsible for detecting an object), 'negative' (background), or 'neutral' (ignored).
    This matching is typically based on Intersection over Union (IoU) thresholds.
    """
    def __init__(self, overlap_threshold_pos=0.5, overlap_threshold_neg=0.4):
        self.overlap_threshold_pos = overlap_threshold_pos
        self.overlap_threshold_neg = overlap_threshold_neg
        self.box_encoder = BoxEncoder()

    def __call__(self, gt_boxes, gt_labels, anchors, device):
        """
        Matches ground truth boxes to anchors.
        Args:
            gt_boxes: (tensor) Ground truth bounding boxes, shape (num_gt, 4).
            gt_labels: (tensor) Ground truth labels, shape (num_gt,).
            anchors: (tensor) Anchor boxes, shape (num_anchors, 4).
            device: (torch.device) Device to perform computations on.
        Returns:
            Tuple[Tensor, Tensor]:
                - loc_targets: Encoded location targets for each anchor, shape (num_anchors, 4).
                - cls_targets: Class targets for each anchor, shape (num_anchors,). 0 for background.
        """
        num_anchors = anchors.size(0)
        num_gt = gt_boxes.size(0)

        # Move everything to the correct device
        gt_boxes = gt_boxes.to(device)
        gt_labels = gt_labels.to(device)
        anchors = anchors.to(device)

        # Initialize targets
        # Location targets (num_anchors, 4) - will store encoded offsets
        loc_targets = torch.zeros((num_anchors, 4), dtype=torch.float32, device=device)
        # Class targets (num_anchors,) - will store 0 for background, 1+ for object classes
        cls_targets = torch.zeros(num_anchors, dtype=torch.long, device=device)

        if num_gt == 0:
            # If no ground truth, all anchors are background (class 0)
            return loc_targets, cls_targets

        # Compute IoU between all anchors and all ground truth boxes
        # iou_matrix shape (num_anchors, num_gt)
        iou_matrix = iou(anchors, gt_boxes)

        # 1. Match each anchor to the GT box with the highest IoU
        # best_gt_iou: (num_anchors,) - max IoU for each anchor with any GT
        # best_gt_idx: (num_anchors,) - index of the GT box that yielded max IoU
        best_gt_iou, best_gt_idx = iou_matrix.max(dim=1)

        # 2. Match each GT box to the anchor with the highest IoU
        # best_anchor_iou: (num_gt,) - max IoU for each GT with any anchor
        # best_anchor_idx: (num_gt,) - index of the anchor that yielded max IoU
        best_anchor_iou, best_anchor_idx = iou_matrix.max(dim=0)

        # Ensure each ground truth box is matched to its best anchor
        # For each GT box, assign its label and encoded box to its best anchor
        for gt_idx, anchor_idx in enumerate(best_anchor_idx):
            cls_targets[anchor_idx] = gt_labels[gt_idx]
            loc_targets[anchor_idx] = self.box_encoder.encode(gt_boxes[gt_idx].unsqueeze(0), anchors[anchor_idx].unsqueeze(0))

        # Match anchors with high IoU to any ground truth box
        # Positive matches: anchors with IoU > overlap_threshold_pos
        pos_mask = best_gt_iou >= self.overlap_threshold_pos
        # For these positive anchors, assign the label and encoded box of their best matched GT
        cls_targets[pos_mask] = gt_labels[best_gt_idx[pos_mask]]
        loc_targets[pos_mask] = self.box_encoder.encode(gt_boxes[best_gt_idx[pos_mask]], anchors[pos_mask])

        # Negative matches: anchors with IoU < overlap_threshold_neg
        # And not already matched as positive
        neg_mask = (best_gt_iou < self.overlap_threshold_neg) & (cls_targets == 0)
        # For these, the class target remains 0 (background) and loc_targets remain 0 (no object to regress)
        cls_targets[neg_mask] = 0

        return loc_targets, cls_targets

logger.info("BoxEncoder and MatchPrior classes defined.")

In [ ]:
#compute the ssd head loss from the outputs
#takes into account the match prior, boxencoder classes

def compute_ssd_loss(
    loc_preds: torch.Tensor, # (B, total_anchors, 4)
    cls_logits: torch.Tensor, # (B, total_anchors, num_classes + 1)
    gt_boxes_batch: List[torch.Tensor],
    gt_labels_batch: List[torch.Tensor],
    anchors: torch.Tensor,
    cls_loss_fn: nn.Module,
    reg_loss_fn: nn.Module,
    device: torch.device,
    neg_pos_ratio: int = 3
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Computes the total SSD loss for a batch, including localization and classification losses.

    Args:
        loc_preds: Predicted localization offsets from the model, shape (B, N_anchors, 4).
        cls_logits: Predicted class logits from the model, shape (B, N_anchors, C+1).
        gt_boxes_batch: List of ground truth boxes for each image in the batch.
        gt_labels_batch: List of ground truth labels for each image in the batch.
        anchors: Pre-computed anchor boxes, shape (N_anchors, 4).
        cls_loss_fn: Classification loss function (e.g., nn.CrossEntropyLoss).
        reg_loss_fn: Regression loss function (e.g., nn.SmoothL1Loss).
        device: The device (CPU or GPU) to perform computations on.
        neg_pos_ratio: The ratio of negative samples to positive samples for hard negative mining.

    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]: A tuple containing the total loss,
        localization loss, and classification loss for the batch.
    """
    logger = get_logger()

    batch_size = loc_preds.size(0)
    num_anchors = anchors.size(0)

    # Initialize MatchPrior
    matcher = MatchPrior()

    total_loc_loss = torch.tensor(0.0, device=device)
    total_cls_loss = torch.tensor(0.0, device=device)

    for i in range(batch_size):
        # Get ground truth for current image
        gt_boxes = gt_boxes_batch[i]
        gt_labels = gt_labels_batch[i]

        # Match ground truth boxes to anchors
        # loc_targets_i: (num_anchors, 4) - encoded offsets for positive matches
        # cls_targets_i: (num_anchors,) - class IDs (0 for background, 1+ for objects)
        loc_targets_i, cls_targets_i = matcher(gt_boxes, gt_labels, anchors, device)

        # --- Localization Loss (L_loc) ---
        # Only calculate localization loss for positive anchors (cls_targets_i > 0)
        pos_mask = cls_targets_i > 0
        if pos_mask.sum() > 0:
            # Predicted locations for positive anchors
            loc_preds_pos = loc_preds[i, pos_mask]
            # Ground truth encoded locations for positive anchors
            loc_targets_pos = loc_targets_i[pos_mask]
            total_loc_loss += reg_loss_fn(loc_preds_pos, loc_targets_pos)
        else:
            # If no positive samples, localization loss is 0 for this image
            total_loc_loss += torch.tensor(0.0, device=device)

        # --- Classification Loss (L_cls) ---
        # Hard Negative Mining:
        # 1. Compute classification loss for all anchors
        # 2. Identify positive anchors (cls_targets_i > 0)
        # 3. Identify negative anchors (cls_targets_i == 0)
        # 4. For negative anchors, sort by loss and pick 'neg_pos_ratio' * num_pos_samples
        # 5. Combine positive and selected hard negative anchors for final classification loss

        # Compute cross-entropy loss for all anchors, but keep it unreduced initially
        # Note: cls_logits[i] is (num_anchors, C+1), cls_targets_i is (num_anchors,)
        all_cls_loss = cls_loss_fn(cls_logits[i], cls_targets_i)

        # Get number of positive anchors
        num_pos = pos_mask.sum()

        if num_pos > 0:
            # Select positive anchors
            pos_cls_loss = all_cls_loss[pos_mask]

            # Select negative anchors (where cls_targets_i == 0)
            neg_mask = (cls_targets_i == 0)
            neg_cls_loss_unfiltered = all_cls_loss[neg_mask]

            # Sort negative losses in descending order and select top ones (hard negatives)
            # Ensure we don't try to select more negatives than available
            num_neg_to_select = min(neg_pos_ratio * num_pos, neg_cls_loss_unfiltered.size(0))

            if num_neg_to_select > 0:
                # Use .topk() to get the largest losses
                hard_neg_cls_loss, _ = neg_cls_loss_unfiltered.topk(num_neg_to_select)
                # Combine positive and hard negative losses
                combined_cls_loss = torch.cat([pos_cls_loss, hard_neg_cls_loss])
            else:
                # Only positive losses if no hard negatives are selected
                combined_cls_loss = pos_cls_loss
        else:
            # If no positive anchors, select hard negatives from all negatives
            neg_mask = (cls_targets_i == 0)
            neg_cls_loss_unfiltered = all_cls_loss[neg_mask]

            num_neg_to_select = min(num_anchors // (neg_pos_ratio + 1), neg_cls_loss_unfiltered.size(0))
            if num_neg_to_select > 0:
                 hard_neg_cls_loss, _ = neg_cls_loss_unfiltered.topk(num_neg_to_select)
                 combined_cls_loss = hard_neg_cls_loss
            else:
                combined_cls_loss = torch.tensor([], device=device) # No pos or neg to learn from

        if combined_cls_loss.numel() > 0:
            total_cls_loss += combined_cls_loss.mean()
        else:
            total_cls_loss += torch.tensor(0.0, device=device)

    # Average losses over the batch
    total_loc_loss /= batch_size
    total_cls_loss /= batch_size

    # Combine total loss
    total_loss = total_loc_loss + total_cls_loss

    logger.info(f"Batch Loss: Total={total_loss.item():.4f}, Loc={total_loc_loss.item():.4f}, Cls={total_cls_loss.item():.4f}")

    return total_loss, total_loc_loss, total_cls_loss

logger.info("compute_ssd_loss function defined with Hard Negative Mining.")

### Define the Training Function

Now, we'll create a `trainer` function that orchestrates the entire training process. This function will:

1.  **Compile the model with JIT (`torch.jit.script`)**: This can lead to performance improvements and enables model deployment in environments without a Python interpreter.
2.  **Iterate through epochs**: For each epoch, it will perform a training pass and an optional validation pass.
3.  **Training Loop**: For each batch in the training DataLoader:
    *   Move data to the appropriate device (CPU/GPU).
    *   Perform a forward pass.
    *   Calculate the total loss using `compute_ssd_loss`.
    *   Perform backpropagation and update model weights using the optimizer.
    *   Update the learning rate scheduler.
4.  **Validation Loop**: After each training epoch, if a validation DataLoader is provided, it will:
    *   Set the model to evaluation mode (`model.eval()`).
    *   Calculate the loss on the validation set without updating weights (`torch.no_grad()`).
    *   Log validation metrics.
5.  **Logging**: Provide insights into the training progress by logging losses per batch and per epoch.

### Define Optimizer and Learning Rate Scheduler

For training, we need an optimizer to update the model's weights and a learning rate scheduler to adjust the learning rate during training, which can help in achieving better convergence.

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from typing import Dict, Any, Tuple

def get_optimizer_and_scheduler(
    model_params: Any,
    optimizer_name: str = 'SGD',
    optimizer_hparams: Dict[str, Any] = None,
    scheduler_name: str = 'StepLR',
    scheduler_hparams: Dict[str, Any] = None,
) -> Tuple[optim.Optimizer, optim.lr_scheduler._LRScheduler, Dict[str, Any], Dict[str, Any]]:
    """
    Initializes an optimizer and a learning rate scheduler.

    Args:
        model_params: The parameters of the model to optimize.
        optimizer_name (str): Name of the optimizer to use (e.g., 'SGD', 'Adam').
        optimizer_hparams (Dict[str, Any]): Dictionary of hyperparameters for the optimizer.
        scheduler_name (str): Name of the scheduler to use (e.g., 'StepLR').
        scheduler_hparams (Dict[str, Any]): Dictionary of hyperparameters for the scheduler.

    Returns:
        Tuple[optim.Optimizer, optim.lr_scheduler._LRScheduler, Dict[str, Any], Dict[str, Any]]:
        The optimizer, scheduler, actual optimizer hyperparameters, and actual scheduler hyperparameters.
    """
    logger = get_logger()

    # Default optimizer hyperparameters
    if optimizer_hparams is None:
        optimizer_hparams = {'lr': 0.005, 'momentum': 0.9, 'weight_decay': 0.0005}
    # Make a copy to ensure we return the actual params used
    actual_optimizer_hparams = optimizer_hparams.copy()

    # Initialize optimizer
    if optimizer_name.lower() == 'sgd':
        optimizer = optim.SGD(model_params, **actual_optimizer_hparams)
    elif optimizer_name.lower() == 'adam':
        optimizer = optim.Adam(model_params, **actual_optimizer_hparams)
    # Add more optimizers as needed
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")
    logger.info(f"Optimizer ({optimizer_name}) initialized with parameters: {actual_optimizer_hparams}")

    # Default scheduler hyperparameters
    if scheduler_hparams is None:
        scheduler_hparams = {'step_size': 3, 'gamma': 0.1}
    # Make a copy to ensure we return the actual params used
    actual_scheduler_hparams = scheduler_hparams.copy()

    # Initialize scheduler
    if scheduler_name.lower() == 'steplr':
        lr_scheduler = StepLR(optimizer, **actual_scheduler_hparams)
    # Add more schedulers as needed
    else:
        logger.warning(f"Unsupported scheduler: {scheduler_name}. No scheduler will be used.")
        lr_scheduler = None

    if lr_scheduler:
        logger.info(f"Learning rate scheduler ({scheduler_name}) initialized with parameters: {actual_scheduler_hparams}")

    return optimizer, lr_scheduler, actual_optimizer_hparams, actual_scheduler_hparams

## Call the optimizer and learning rate scheduler


In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

# Initialize optimizer and scheduler using the new function
optimizer, lr_scheduler, optimizer_hparams_used, scheduler_hparams_used = get_optimizer_and_scheduler(
    params,
    optimizer_name='SGD',
    optimizer_hparams={'lr': 0.005, 'momentum': 0.9, 'weight_decay': 0.0005},
    scheduler_name='StepLR',
    scheduler_hparams={'step_size': 3, 'gamma': 0.1}
)
logger.info("Optimizer and scheduler setup completed.")

In [ ]:
params

## Trainer

In [ ]:
import torch.jit as jit

def trainer(
    model: nn.Module,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    optimizer: optim.Optimizer,
    lr_scheduler: optim.lr_scheduler._LRScheduler,
    cls_loss_fn: nn.Module,
    reg_loss_fn: nn.Module,
    compute_ssd_loss_fn,
    device: torch.device,
    num_epochs: int,
    run_name: str = "obj_det_run", #set to any name you want
    optimizer_hparams: Dict[str, Any] = None,
    scheduler_hparams: Dict[str, Any] = None,
):
    logger = get_logger()

    # Setup TensorBoard SummaryWriter
    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(
        'runs',
        f"{run_name}_{current_time}_epochs-{num_epochs}"
    )
    writer = SummaryWriter(log_dir=log_dir)
    logger.info(f"TensorBoard logs will be saved to: {log_dir}")

    # Compile the model with JIT
    # The model should be on the target device before scripting
    try:
        scripted_model = jit.script(model)
        logger.info("Model successfully compiled with TorchScript JIT.")
    except Exception as e:
        logger.error(f"Failed to compile model with TorchScript JIT: {e}")
        scripted_model = model # Fallback to unscripted model

    scripted_model.to(device) # Ensure the (possibly scripted) model is on the correct device

    global_step = 0
    for epoch in range(num_epochs):
        scripted_model.train()  # Set model to training mode
        total_train_loss = 0.0
        total_train_loc_loss = 0.0
        total_train_cls_loss = 0.0

        logger.info(f"\nEpoch {epoch+1}/{num_epochs} - Training...")
        for batch_idx, (images, gt_boxes_batch, gt_labels_batch) in enumerate(train_dataloader):
            images = images.to(device)
            # gt_boxes_batch and gt_labels_batch are lists of tensors, so they don't need .to(device) here
            # they are handled inside compute_ssd_loss_fn

            optimizer.zero_grad()

            # Forward pass to get raw predictions (loc_preds, cls_logits)
            outputs = scripted_model(images)
            loc_preds = outputs['loc_preds']
            cls_logits = outputs['cls_logits']
            anchors = scripted_model.anchors # Access anchors from the model buffer

            # Compute loss
            loss, loc_loss, cls_loss = compute_ssd_loss_fn(
                loc_preds, cls_logits, gt_boxes_batch, gt_labels_batch,
                anchors, cls_loss_fn, reg_loss_fn, device
            )

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            total_train_loc_loss += loc_loss.item()
            total_train_cls_loss += cls_loss.item()

            # Log batch metrics to TensorBoard
            writer.add_scalar('Loss/train_batch', loss.item(), global_step=global_step)
            writer.add_scalar('Loss/train_loc_batch', loc_loss.item(), global_step=global_step)
            writer.add_scalar('Loss/train_cls_batch', cls_loss.item(), global_step=global_step)

            if (batch_idx + 1) % 10 == 0:
                logger.info(
                    f"  Batch {batch_idx+1}/{len(train_dataloader)}: "
                    f"Loss: {loss.item():.4f} (Loc: {loc_loss.item():.4f}, Cls: {cls_loss.item():.4f})"
                )
            global_step += 1

        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_train_loc_loss = total_train_loc_loss / len(train_dataloader)
        avg_train_cls_loss = total_train_cls_loss / len(train_dataloader)
        logger.info(
            f"Epoch {epoch+1} Training Summary: Avg Loss: {avg_train_loss:.4f} "
            f"(Avg Loc: {avg_train_loc_loss:.4f}, Avg Cls: {avg_train_cls_loss:.4f})"
        )

        # Log epoch-level training metrics
        writer.add_scalar('Loss/train_epoch', avg_train_loss, global_step=epoch)
        writer.add_scalar('Loss/train_loc_epoch', avg_train_loc_loss, global_step=epoch)
        writer.add_scalar('Loss/train_cls_epoch', avg_train_cls_loss, global_step=epoch)

        if lr_scheduler:
            lr_scheduler.step()
            current_lr = optimizer.param_groups[0]['lr']
            logger.info(f"  Learning rate updated to: {current_lr:.6f}")
            writer.add_scalar('Learning_Rate', current_lr, global_step=epoch)

        # --- Validation ---
        if val_dataloader:
            scripted_model.eval()  # Set model to evaluation mode
            total_val_loss = 0.0
            total_val_loc_loss = 0.0
            total_val_cls_loss = 0.0
            logger.info(f"Epoch {epoch+1}/{num_epochs} - Validating...")
            with torch.no_grad():
                for batch_idx, (images, gt_boxes_batch, gt_labels_batch) in enumerate(val_dataloader):
                    images = images.to(device)
                    outputs = scripted_model(images)
                    loc_preds = outputs['loc_preds']
                    cls_logits = outputs['cls_logits']
                    anchors = scripted_model.anchors

                    loss, loc_loss, cls_loss = compute_ssd_loss_fn(
                        loc_preds, cls_logits, gt_boxes_batch, gt_labels_batch,
                        anchors, cls_loss_fn, reg_loss_fn, device
                    )

                    total_val_loss += loss.item()
                    total_val_loc_loss += loc_loss.item()
                    total_val_cls_loss += cls_loss.item()

            avg_val_loss = total_val_loss / len(val_dataloader)
            avg_val_loc_loss = total_val_loc_loss / len(val_dataloader)
            avg_val_cls_loss = total_val_cls_loss / len(val_dataloader)
            logger.info(
                f"Epoch {epoch+1} Validation Summary: Avg Loss: {avg_val_loss:.4f} "
                f"(Avg Loc: {avg_val_loc_loss:.4f}, Avg Cls: {avg_val_cls_loss:.4f})"
            )

            # Log epoch-level validation metrics
            writer.add_scalar('Loss/val_epoch', avg_val_loss, global_step=epoch)
            writer.add_scalar('Loss/val_loc_epoch', avg_val_loc_loss, global_step=epoch)
            writer.add_scalar('Loss/val_cls_epoch', avg_val_cls_loss, global_step=epoch)

    # Log hyperparameters to TensorBoard
    hparams = {
        'num_epochs': num_epochs,
        **{f'optimizer_{k}': v for k, v in (optimizer_hparams or {}).items()},
        **{f'scheduler_{k}': v for k, v in (scheduler_hparams or {}).items()},
        'model_input_size': model.input_size,
        'model_num_classes': model.num_classes
    }
    # Create dummy metrics for hparams.add_hparams
    metric_dict = {
        'hparam/train_loss': avg_train_loss,
        'hparam/val_loss': avg_val_loss if val_dataloader else 0.0
    }
    writer.add_hparams(hparams, metric_dict)

    writer.close()
    logger.info("Training complete!")

Now let's kick off the training process! We'll use a small number of epochs for demonstration. You can adjust `num_epochs` for longer training runs.

In [ ]:
# Define training parameters
num_epochs = 100 # Set a small number of epochs for initial testing
run_name = "initial_tiny_resnet_ssd" # A descriptive name for your run

logger.info(f"Starting training for {num_epochs} epochs...")

# Call the trainer function
trainer(
    model=model,  # The original unscripted model, it will be scripted inside trainer
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    cls_loss_fn=cls_loss_fn,
    reg_loss_fn=reg_loss_fn,
    compute_ssd_loss_fn=compute_ssd_loss,
    device=device,
    num_epochs=num_epochs,
    run_name=run_name, # Pass the run name
    optimizer_hparams=optimizer_hparams_used, # Pass the actual optimizer hparams
    scheduler_hparams=scheduler_hparams_used # Pass the actual scheduler hparams
)

logger.info("Training process initiated.")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
#utlity function to save the trained model fully or the statedict
def save_model(
    model: nn.Module,
    run_name: str,
    num_epochs: int,
    save_full_model: bool = False,
    output_dir: str = 'saved_models'
):
    """
    Saves the model or its state dictionary to a specified directory.

    Args:
        model (nn.Module): The PyTorch model to save.
        run_name (str): The name of the current training run, used for naming the saved file.
        num_epochs (int): The number of epochs the model was trained for.
        save_full_model (bool): If True, saves the entire model (including architecture);
                                if False, saves only the state dictionary (weights).
        output_dir (str): The directory where the model will be saved.
    """
    logger = get_logger()
    os.makedirs(output_dir, exist_ok=True)

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if save_full_model:
        filename = f"{output_dir}/{run_name}_full_model_epochs-{num_epochs}_{current_time}.pt"
        torch.save(model, filename)
        logger.info(f"Full model saved to {filename}")
    else:
        filename = f"{output_dir}/{run_name}_state_dict_epochs-{num_epochs}_{current_time}.pth"
        torch.save(model.state_dict(), filename)
        logger.info(f"Model state dictionary saved to {filename}")

In [ ]:
# Save the trained model's state dictionary
save_model(
    model=model, # Pass the original unscripted model here
    run_name=run_name,
    num_epochs=num_epochs,
    save_full_model=False, # Set to True to save the full model, False for state dict only
    output_dir='saved_models'
)

logger.info("Model saving process initiated.")

In [ ]:
import glob
def load_model_from_disk(
    output_dir: str,
    run_name: str,
    num_classes: int,
    device: torch.device,
    model_class: nn.Module = ObjDet_V1
):
    logger = get_logger()

    # Search for full model (.pt) files first
    full_model_files = glob.glob(f'{output_dir}/{run_name}_full_model_epochs-*.pt')
    if full_model_files:
        latest_full_model = max(full_model_files, key=os.path.getctime)
        logger.info(f"Loading full model from: {latest_full_model}")
        try:
            model = torch.load(latest_full_model, map_location=device)
            model.eval() # Set to evaluation mode
            logger.info("Full model loaded successfully.")
            return model
        except Exception as e:
            logger.error(f"Error loading full model {latest_full_model}: {e}")

    # If no full model, search for state dictionary (.pth) files
    state_dict_files = glob.glob(f'{output_dir}/{run_name}_state_dict_epochs-*.pth')
    if state_dict_files:
        latest_state_dict = max(state_dict_files, key=os.path.getctime)
        logger.info(f"Loading model state dictionary from: {latest_state_dict}")
        try:
            # Create a new instance of the model (ensure it's on the correct device)
            model = model_class(num_classes=num_classes).to(device)
            # Load the state dictionary
            model.load_state_dict(torch.load(latest_state_dict, map_location=device))
            model.eval() # Set to evaluation mode
            logger.info("Model state dictionary loaded successfully.")
            return model
        except Exception as e:
            logger.error(f"Error loading model state dictionary {latest_state_dict}: {e}")

    logger.error(f"No saved models (.pt or .pth) found for run_name '{run_name}' in {output_dir}.")
    return None

logger.info("load_model_from_disk function defined.")

### Load Model for Display

Now, we'll use the `load_model_from_disk` function to load the most recently saved model. This model, whether a full model or just its state dictionary, will then be used to display predictions on a sample image.

In [ ]:
# Define the path to the saved model directory
saved_models_dir = 'saved_models'

# Load the latest saved model
loaded_model = load_model_from_disk(
    output_dir=saved_models_dir,
    run_name=run_name, # Use the run_name from training parameters
    num_classes=len(class_names), # Pass the number of object classes
    device=device,
    model_class=ObjDet_V1 # Pass the model class for state_dict loading
)

if loaded_model:
    logger.info("Model loaded successfully, proceeding to display predictions.")
    # Now, use the loaded_model for predictions
    display_model_predictions(
        model=loaded_model,
        annotated_images=train_annotations,
        class_names=class_names, # pass class_names for proper labeling
        num_images_to_display=1,
        confidence_threshold=0.1, # Adjust as needed, e.g., 0.0 for untrained model
        show_ground_truth=True,
        apply_nms=True,
        nms_iou_threshold=0.2
    )
else:
    logger.error("Failed to load any model. Cannot display predictions.")

In [ ]:
!pip install torchmetrics -qU

In [ ]:
pip install thop -qU

### Object Detection Metrics Calculator Class

This class encapsulates methods for calculating key object detection metrics such as Mean Average Precision (mAP) and Floating Point Operations (FLOPs). It provides a structured way to evaluate model performance.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision
from thop.profile import profile
import torch.nn as nn
import torch
from typing import Dict, List

class ObjectDetectionMetricsCalculator:
    """
    A class to encapsulate methods for calculating various object detection metrics.
    """
    def __init__(self, model: nn.Module, device: torch.device, class_names: List[str]):
        self.model = model
        self.device = device
        self.class_names = class_names
        self.logger = get_logger()
        # Ensure model is on the correct device
        self.model.to(self.device)
        self.model.eval() # Set model to evaluation mode by default

    def calculate_map(self,
                      dataloader: DataLoader,
                      confidence_threshold: float = 0.001,
                      iou_threshold: float = 0.5) -> Dict[str, torch.Tensor]:
        """
        Calculates Mean Average Precision (mAP) for the object detection model.

        Args:
            dataloader (DataLoader): DataLoader for the dataset (e.g., validation set).
            confidence_threshold (float): Minimum confidence score to consider a prediction.
            iou_threshold (float): IoU threshold for non-maximum suppression (NMS) during evaluation.

        Returns:
            Dict[str, torch.Tensor]: A dictionary containing mAP metrics.
        """
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=True, iou_thresholds=[iou_threshold])

        all_preds = []
        all_targets = []

        self.logger.info("Calculating mAP. Iterating through dataloader...")
        with torch.no_grad():
            for images, gt_boxes_batch, gt_labels_batch in dataloader:
                images = images.to(self.device)

                # Model predictions
                predictions = self.model(images)

                for i in range(images.shape[0]):
                    # Process predictions for current image
                    pred_boxes = predictions['boxes'][i]
                    pred_labels_raw = predictions['labels'][i]
                    pred_scores = predictions['scores'][i]

                    # Filter by confidence threshold
                    keep_preds = pred_scores > confidence_threshold
                    pred_boxes = pred_boxes[keep_preds]
                    pred_labels_raw = pred_labels_raw[keep_preds]
                    pred_scores = pred_scores[keep_preds]

                    # Apply NMS on predictions
                    if pred_boxes.numel() > 0:
                        nms_keep_indices = nms(pred_boxes, pred_scores, iou_threshold)
                        pred_boxes = pred_boxes[nms_keep_indices]
                        pred_labels_raw = pred_labels_raw[nms_keep_indices]
                        pred_scores = pred_scores[nms_keep_indices]

                    # Filter out background predictions (label 0) for mAP calculation
                    # Map 1-indexed model labels to 0-indexed for torchmetrics
                    valid_preds = pred_labels_raw > 0
                    formatted_preds = {
                        "boxes": pred_boxes[valid_preds],
                        "scores": pred_scores[valid_preds],
                        "labels": pred_labels_raw[valid_preds] - 1 # Convert to 0-indexed
                    }
                    all_preds.append(formatted_preds)

                    # Process ground truth for current image
                    # Map 1-indexed ground truth labels to 0-indexed for torchmetrics
                    formatted_target = {
                        "boxes": gt_boxes_batch[i].to(self.device),
                        "labels": gt_labels_batch[i].to(self.device) - 1 # Convert to 0-indexed
                    }
                    all_targets.append(formatted_target)

        # Update the metric with predictions and targets
        metric.update(all_preds, all_targets)

        # Compute and return the results
        results = metric.compute()
        self.logger.info("mAP calculation complete.")
        return results

    def calculate_flops(self, input_shape: Tuple[int, int, int, int] = (1, 3, 512, 512)) -> float:
        """
        Calculates the Floating Point Operations (FLOPs) for the model.

        Args:
            input_shape (Tuple[int, int, int, int]): The shape of a dummy input tensor
                                                      (batch_size, channels, height, width).

        Returns:
            float: The number of FLOPs in GigaFLOPs (GFLOPs).
        """
        self.logger.info(f"Calculating FLOPs for input shape: {input_shape}")
        dummy_input = torch.randn(input_shape).to(self.device)

        # Use thop.profile to get FLOPs and parameters
        # verbose=False to suppress detailed layer-wise output
        total_ops, total_params = profile(self.model, (dummy_input,), verbose=False)

        # Convert FLOPs from operations to GFLOPs (1 GFLOP = 10^9 operations)
        gflops = total_ops / 1e9
        self.logger.info(f"FLOPs: {gflops:.2f} GFLOPs")
        self.logger.info(f"Number of parameters: {total_params / 1e6:.2f} M")
        return gflops

    def display_map_results(self, map_results: Dict[str, torch.Tensor]):
        """
        Displays the mAP results in a formatted way.
        """
        self.logger.info("Mean Average Precision results:")
        for k, v in map_results.items():
            if isinstance(v, torch.Tensor):
                if v.numel() == 1: # Check if it's a scalar tensor
                    self.logger.info(f"{k}: {v.item():.4f}")
                else: # Handle per-class tensors like 'map_per_class' or 'mar_100_per_class'
                    self.logger.info(f"{k}:")
                    for class_idx, val in enumerate(v):
                        if class_idx < len(self.class_names):
                            self.logger.info(f"  Class '{self.class_names[class_idx]}': {val.item():.4f}")
                        else:
                            self.logger.info(f"  Unknown Class {class_idx}: {val.item():.4f}")
            else:
                self.logger.info(f"{k}: {v}")


logger.info("ObjectDetectionMetricsCalculator class defined.")

### Use the `ObjectDetectionMetricsCalculator` to get metrics

In [ ]:
# Instantiate the metrics calculator
metrics_calculator = ObjectDetectionMetricsCalculator(model=loaded_model, device=device, class_names=class_names)

# Calculate and display mAP results
map_results = metrics_calculator.calculate_map(
    dataloader=val_dataloader,
    confidence_threshold=0.1,
    iou_threshold=0.2
)
metrics_calculator.display_map_results(map_results)

# Calculate FLOPs
flops = metrics_calculator.calculate_flops(input_shape=(1, 3, model.input_size, model.input_size))

logger.info("Metrics calculation complete.")

In [ ]:
print(model.state_dict())

In [ ]:
num_epochs = 200
run_name = "tuned_tiny_resnet_ssd" # new run name

# Define common optimizer hyperparameters for object detection
optimizer_hparams_new = {'lr': 0.001, 'momentum': 0.9, 'weight_decay': 0.0001}
# Define common scheduler hyperparameters
scheduler_hparams_new = {'step_size': 50, 'gamma': 0.1} # StepLR to reduce LR every 50 epochs

logger.info(f"Starting training for {num_epochs} epochs with tuned hyperparameters...")

# Initialize optimizer and scheduler with new parameters
params = [p for p in model.parameters() if p.requires_grad]
optimizer, lr_scheduler, optimizer_hparams_used, scheduler_hparams_used = get_optimizer_and_scheduler(
    params,
    optimizer_name='SGD',
    optimizer_hparams=optimizer_hparams_new,
    scheduler_name='StepLR',
    scheduler_hparams=scheduler_hparams_new
)
logger.info("Optimizer and scheduler re-setup completed with new parameters.")

# Call the trainer function with the new parameters
trainer(
    model=model,  # The original unscripted model, it will be scripted inside trainer
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    cls_loss_fn=cls_loss_fn,
    reg_loss_fn=reg_loss_fn,
    compute_ssd_loss_fn=compute_ssd_loss,
    device=device,
    num_epochs=num_epochs,
    run_name=run_name, # Pass the new run name
    optimizer_hparams=optimizer_hparams_used, # Pass the actual optimizer hparams
    scheduler_hparams=scheduler_hparams_used # Pass the actual scheduler hparams
)

logger.info("Training process initiated with new hyperparameters.")

In [ ]:
from torch.utils import tensorboard
%reload_ext tensorboard
%tensorboard --logdir runs

In [ ]:
save_model(
    model=model, # Pass the original unscripted model here
    run_name=run_name,
    num_epochs=num_epochs,
    save_full_model=False, # Saving the state dict is generally preferred
    output_dir='saved_models'
)

logger.info("Model saving process initiated.")

In [ ]:
# Define the path to the saved model directory
saved_models_dir = 'saved_models'

# Load the latest saved model using the run_name from the tuned training
loaded_tuned_model = load_model_from_disk(
    output_dir=saved_models_dir,
    run_name=run_name, # Use the run_name from the tuned training parameters
    num_classes=len(class_names), # Pass the number of object classes
    device=device,
    model_class=ObjDet_V1 # Pass the model class for state_dict loading
)

In [ ]:
if loaded_tuned_model:
    logger.info("Tuned model loaded successfully, proceeding to display predictions.")
    # Display predictions using the loaded tuned model
    display_model_predictions(
        model=loaded_tuned_model,
        annotated_images=train_annotations,
        class_names=class_names,
        num_images_to_display=1,
        confidence_threshold=0.1,
        show_ground_truth=True,
        apply_nms=True,
        nms_iou_threshold=0.1
    )
else:
    logger.error("Failed to load the tuned model. Cannot display predictions.")

In [ ]:
# Instantiate the metrics calculator if not already done (e.g., if this cell is run independently)
if 'metrics_calculator' not in locals() or metrics_calculator is None:
    logger.info("Instantiating ObjectDetectionMetricsCalculator...")
    metrics_calculator = ObjectDetectionMetricsCalculator(model=loaded_tuned_model, device=device, class_names=class_names)

# Calculate and display mAP results
logger.info("Starting mAP calculation...")
map_results = metrics_calculator.calculate_map(
    dataloader=val_dataloader, # Use the validation dataloader
    confidence_threshold=0.1,
    iou_threshold=0.2
)
metrics_calculator.display_map_results(map_results)

# Calculate FLOPs
logger.info("Starting FLOPs calculation...")
flops = metrics_calculator.calculate_flops(input_shape=(1, 3, model.input_size, model.input_size))
logger.info(f"FLOPs: {flops:.2f} GFLOPs")

logger.info("Metrics calculation complete.")